# Chain-of-Thought with LangGraph + a local LLM
Runtime → Change runtime type → **T4 GPU**, then run all cells.
Graph: `plan → reason → answer`. Each node is one LLM call, so the chain of thought is explicit and inspectable.

In [ ]:
!pip -q install langgraph transformers accelerate

## 1. Load a small open LLM (no API key needed)

In [ ]:
import torch
from transformers import pipeline

gen = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
)

def llm(system: str, user: str, max_new_tokens: int = 400) -> str:
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": user}]
    out = gen(messages, max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"][-1]["content"].strip()

## 2. Define the graph state and nodes

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    plan: str
    reasoning: str
    answer: str

def plan_node(state: State):
    plan = llm(
        "Break the problem into 2-4 short numbered steps. Do NOT solve it yet.",
        state["question"],
    )
    return {"plan": plan}

def reason_node(state: State):
    reasoning = llm(
        "Solve the problem step by step, following the plan. Show your work. "
        "End with a line: Final answer: <answer>",
        f"Question: {state['question']}\n\nPlan:\n{state['plan']}",
        max_new_tokens=500,
    )
    return {"reasoning": reasoning}

def answer_node(state: State):
    text = state["reasoning"]
    final = text.split("Final answer:")[-1].strip() if "Final answer:" in text else text.splitlines()[-1]
    return {"answer": final}

## 3. Build and compile the graph

In [ ]:
builder = StateGraph(State)
builder.add_node("plan", plan_node)
builder.add_node("reason", reason_node)
builder.add_node("answer", answer_node)

builder.add_edge(START, "plan")
builder.add_edge("plan", "reason")
builder.add_edge("reason", "answer")
builder.add_edge("answer", END)

graph = builder.compile()

## 4. Run it

In [ ]:
question = "A train travels 60 km/h for 2.5 hours, then 80 km/h for 1.5 hours. What is the total distance?"

result = graph.invoke({"question": question})

print("PLAN:\n", result["plan"])
print("\nREASONING:\n", result["reasoning"])
print("\nANSWER:", result["answer"])

## 5. (Optional) Visualize the graph

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))